# Fine-tuning Cellpose on the annotation rounds

Trains a new Cellpose model from `cpsam_v2` on the ROIs corrected in AnnoCell, then segments the store with it, as
one more segmentation (see `macroseg.train`).

- **ROIs:** every complete ROI under `$MACROSEG_ANNOTATIONS` (every cell checked, `annocell.folder.is_complete`),
  read from its store with the channels of the segmentation it started from (`membrane_dapi`: NaK_ATPase + DAPI),
  cut into tiles of `TILE_PX`.
- **No validation split:** every tile is used for training; models are scored on the test regions.
- **Settings** from the config's `training` section, the same every round.
- **Saved** in `$MACROSEG_MODELS/<NAME>/`: the weights, `tiles.csv` (every tile) and `model.json` (start model,
  channels, ROIs, settings, losses, versions).

About 10 to 20 s per epoch on the Mac GPU with 16 tiles of 512 px.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from spatialdata import read_zarr

from macroseg import paths
from macroseg.config import load_config
from macroseg.train import apply_model, load_model_record, train, training_set

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = load_config(ROOT / "configs" / "exp201_test_crop.yaml")

NAME = f"{cfg.name}_r01"        # model name: folder in $MACROSEG_MODELS and labels element in the store
START_MODEL = "cpsam_v2"
TILE_PX = 512
CHANNELS = None                 # None: the channels of the starting segmentation

## 1. Training set
One row per tile. `hand_drawn`: cells drawn or edited in AnnoCell (a cell cut by a tile edge counts in both tiles).

In [ ]:
data = training_set(paths.path("MACROSEG_ANNOTATIONS"), channels=CHANNELS, tile_px=TILE_PX)
print("channels:", data.channels)
data.tiles.groupby(["round", "annotator", "image"])[["cells", "hand_drawn"]].agg(["count", "sum"])

## 2. Train
Epochs, learning rate, weight decay and batch size from the config (`training`).

In [ ]:
folder = train(data, paths.path("MACROSEG_MODELS"), NAME, cfg.training, start_model=START_MODEL)
folder

In [ ]:
record = load_model_record(folder)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(record["train_losses"], linewidth=2)
ax.set_xlabel("epoch")
ax.set_ylabel("train loss")
ax.spines[["top", "right"]].set_visible(False);

## 3. Segment the store with the new model
Adds the labels element `NAME` and its table `<NAME>_intensities` to the store.

In [6]:
sdata = read_zarr(cfg.zarr)
sdata = apply_model(sdata, cfg.image.element, folder, labels=NAME)
sdata

no parent found for <ome_zarr.reader.Label object at 0x121455010>: None
no parent found for <ome_zarr.reader.Label object at 0x1380fd480>: None
no parent found for <ome_zarr.reader.Label object at 0x1380fd6a0>: None
no parent found for <ome_zarr.reader.Label object at 0x138113250>: None
no parent found for <ome_zarr.reader.Label object at 0x138113450>: None
2026-10-23 11:18:41.634 | INFO     | harpy.image.segmentation._segmentation:_precondition:440 - Provided array is chunked in 'z' and/or 'c' dimension. Will rechunk in 'z' and/or 'c' dimension.
/Users/massimoz/Documents/macroseg/.venv/lib/python3.13/site-packages/cellpose/dynamics.py:541: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internall

SpatialData object, with associated Zarr store: /Users/massimoz/Documents/macroseg/data/runs/exp201_test_crop.zarr
├── Images
│     └── 'macsima_image': DataArray[cyx] (6, 2048, 2048)
├── Labels
│     ├── 'exp201_test_crop_r01': DataArray[yx] (2048, 2048)
│     ├── 'membrane': DataArray[yx] (2048, 2048)
│     ├── 'membrane_dapi': DataArray[yx] (2048, 2048)
│     ├── 'membrane_dapi_f480': DataArray[yx] (2048, 2048)
│     ├── 'nuclei': DataArray[yx] (2048, 2048)
│     └── 'nuclei_expanded': DataArray[yx] (2048, 2048)
└── Tables
      ├── 'exp201_test_crop_r01_intensities': AnnData (1696, 6)
      ├── 'membrane_dapi_f480_intensities': AnnData (1835, 6)
      ├── 'membrane_dapi_intensities': AnnData (1587, 6)
      ├── 'membrane_intensities': AnnData (1441, 6)
      ├── 'nuclei_expanded_intensities': AnnData (1911, 6)
      └── 'nuclei_intensities': AnnData (1911, 6)
with coordinate systems:
    ▸ 'global', with elements:
        macsima_image (Images), exp201_test_crop_r01 (Labels), membr